# SpectraAE: 1D Conv Autoencoder for LAMOST CN-Star Detection

**Target**: Use unsupervised autoencoders to compress LAMOST spectra (700px -> 64/256 dim),
extract bottleneck features, then classify CN-enhanced stars with PU-Bagging.
Compare different bottleneck dimensions in preserving CN band information.

**CN Molecular Bands**: CN3839 (3830-3883A), CN4142 (4120-4216A), CH4300 (4285-4315A)

**Core Question**: Raw spectra 700px + PU-Bagging achieves PR-AUC=0.848.
Can AE-compressed features retain the CN band discriminative information?
Does increasing bottleneck from 64d to 256d help?

In [ ]:
import sys, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

_PROJECT_ROOT = Path().resolve()
if str(_PROJECT_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT.parent))

from SpectraAE.models.autoencoder import ConvAutoencoder, Encoder, Decoder
from SpectraAE.extract_features import extract_features
from ML.pu_bagging import load_pu_data, run_pu_bagging, build_comparison_df
from ML.utils import compute_cluster_zscore, FEATURE_COLS_CN9

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Data Overview

Loading preprocessed 33,589 LAMOST spectra from `ML/_cache`.
Each spectrum: 700 pixels (lambda 3800-4500A, step 1A), continuum-normalized to flux~1.0.

**73 known CN-enhanced positives** (from CNstar.csv + FT_cands.csv), rest 33,516 unlabeled.

In [ ]:
CACHE_DIR = Path('ML/_cache')
X_clean = np.load(CACHE_DIR / 'X_clean.npy').astype(np.float32)
stars_clean = pd.read_pickle(CACHE_DIR / 'stars_clustered.pkl')
wave = np.arange(3800.0, 4500.0, 1.0)

data_pu = load_pu_data()
y_all = data_pu['y_all']
cluster_ids = data_pu['cluster_ids']
df_model = data_pu['df_model']
pos_mask = y_all == 1
n_pos = int(y_all.sum())

print(f'Spectra shape: {X_clean.shape}')
print(f'Pixel range: [{X_clean.min():.4f}, {X_clean.max():.4f}]')
print(f'Global stats: mean={X_clean.mean():.4f}, std={X_clean.std():.4f}')
print(f'Labels: {len(y_all):,} stars, {n_pos} known CN + {int((~pos_mask).sum()):,} unlabeled')

### 1.1 Spectrum Preview: CN Stars vs Normal Stars

The three CN/CH bands are highlighted. CN-enhanced stars show deeper absorption in these regions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(wave, X_clean[~pos_mask].mean(axis=0), alpha=0.8, lw=1,
        color='#3498db', label=f'Unlabeled (n={(~pos_mask).sum():,})')
ax.plot(wave, X_clean[pos_mask].mean(axis=0), alpha=0.9, lw=1.5,
        color='#e74c3c', label=f'Known CN (n={n_pos})')
ax.axvspan(3830, 3883, alpha=0.1, color='#3498db', label='CN3839')
ax.axvspan(4120, 4216, alpha=0.1, color='#2ecc71', label='CN4142')
ax.axvspan(4285, 4315, alpha=0.1, color='#e67e22', label='CH4300')
ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Normalized Flux')
ax.set_title('Mean Spectra: CN vs Normal')
ax.legend(fontsize=8, loc='lower right')
ax.grid(alpha=0.2)

ax = axes[1]
diff = X_clean[~pos_mask].mean(axis=0) - X_clean[pos_mask].mean(axis=0)
ax.plot(wave, diff, lw=1.2, color='#8e44ad')
ax.fill_between(wave, 0, diff, alpha=0.3, color='#8e44ad')
ax.axvspan(3830, 3883, alpha=0.1, color='#3498db')
ax.axvspan(4120, 4216, alpha=0.1, color='#2ecc71')
ax.axvspan(4285, 4315, alpha=0.1, color='#e67e22')
ax.axhline(y=0, color='black', lw=0.5)
ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Flux Difference')
ax.set_title('Mean Difference: Normal - CN (CN absorption excess)')
ax.grid(alpha=0.2)

fig.suptitle('LAMOST Spectra Overview - CN Band Identification', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2. Autoencoder Model Architecture

**Encoder**: 3-layer 1D Conv (stride=2) downsampling + AdaptiveAvgPool + FC -> bottleneck
**Decoder**: FC + 4-stage linear interpolation upsampling + 1D Conv -> 700px reconstruction

Two variants: AE-64d (lightweight, 10.9x compression) vs AE-256d (expressive, 2.7x compression)

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_summary(name, in_ch, base_ch, latent_dim):
    m = ConvAutoencoder(in_channels=in_ch, base_ch=base_ch, latent_dim=latent_dim)
    n = count_params(m)
    comp = 700 / latent_dim
    print(f'{name:12s} | base_ch={base_ch:2d} latent={latent_dim:3d} |'
          f'{n:>8,} params | compression 700->{latent_dim} ({comp:.1f}x)')
    return m

print('='*70)
print(f'{"Model":12s} | {"Config":20s} | {"Params":>10s} | Compression')
print('='*70)
ae64 = model_summary('AE-64d', 1, 32, 64)
ae256 = model_summary('AE-256d', 1, 64, 256)
print('='*70)

### 2.1 Encoder Shape Progression

Trace the tensor shape through each layer of the encoder.

In [ ]:
def trace_encoder(enc, x_dummy, name):
    print(f'\n{name}:' if name else '')
    shapes = [('Input', tuple(x_dummy.shape))]
    x = enc.pad(x_dummy)
    shapes.append(('ReflectionPad', tuple(x.shape)))
    for i, conv in enumerate([enc.conv1, enc.conv2, enc.conv3], 1):
        x = conv(x)
        shapes.append((f'Conv{i} (s2)', tuple(x.shape)))
    x = enc.pool(x)
    shapes.append(('AdaptiveAvgPool', tuple(x.shape)))
    x = x.flatten(1)
    shapes.append(('Flatten', tuple(x.shape)))
    z = enc.fc(x)
    shapes.append(('FC -> Latent z', tuple(z.shape)))
    for label, s in shapes:
        print(f'  {label:20s} {str(s):20s}')

trace_encoder(ae64.encoder, torch.randn(1, 1, 700), 'AE-64d')
trace_encoder(ae256.encoder, torch.randn(1, 1, 700), 'AE-256d')

## 3. Pretraining: Unsupervised Spectrum Reconstruction

- **Loss**: MSE (per-pixel reconstruction error)
- **Optimizer**: AdamW (lr=1e-3, weight_decay=1e-5)
- **Scheduler**: CosineAnnealingLR (T_max=300, eta_min=1e-6)
- **Early Stopping**: patience=40/50 epochs, monitoring val loss
- **Preprocessing**: Global p1-p99 clipping + scalar standardization

Training was run on RTX 4060 Laptop GPU.

### 3.1 Load Trained Checkpoints

In [ ]:
def load_ae_checkpoint(ckpt_path, latent_dim, base_ch=32):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = ConvAutoencoder(in_channels=1, base_ch=base_ch, latent_dim=latent_dim).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    info = {
        'epoch': ckpt['epoch'], 'val_loss': ckpt['val_loss'],
        'scaler_mean': ckpt['scaler_mean'], 'scaler_std': ckpt['scaler_std'],
        'params': sum(p.numel() for p in model.parameters()),
    }
    return model, info

# AE-64d (trained previously)
model64, info64 = load_ae_checkpoint('SpectraAE/checkpoints/ae_best.pt', latent_dim=64)
print(f'AE-64d:  epoch={info64["epoch"]}  val_loss={info64["val_loss"]:.6f}  params={info64["params"]:,}')

# AE-256d (newly trained)
ckpt256_path = Path('SpectraAE/checkpoints/ae256/ae_best.pt')
has_256 = ckpt256_path.exists()
if has_256:
    model256, info256 = load_ae_checkpoint(ckpt256_path, latent_dim=256, base_ch=64)
    print(f'AE-256d: epoch={info256["epoch"]}  val_loss={info256["val_loss"]:.6f}  params={info256["params"]:,}')
else:
    print('AE-256d: not trained yet')

### 3.2 Training Curves (AE-256d)

In [ ]:
hist_path = Path('SpectraAE/_cache/ae256_history.pkl')
if hist_path.exists():
    with open(hist_path, 'rb') as f:
        hist256 = pickle.load(f)
    print(f'Best epoch: {hist256["best_epoch"]}, val_loss={hist256["best_val_loss"]:.6f},'
          f' time={hist256["elapsed_seconds"]:.0f}s')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    
    ax = axes[0]
    epochs = range(1, len(hist256['train_losses']) + 1)
    ax.plot(epochs, hist256['train_losses'], lw=1, alpha=0.7, color='#3498db', label='Train')
    ax.plot(epochs, hist256['val_losses'], lw=1.5, color='#e74c3c', label='Val')
    ax.axvline(x=hist256['best_epoch'], color='green', linestyle='--', alpha=0.6,
              label=f'Best epoch={hist256["best_epoch"]}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('AE-256d Training History')
    ax.legend()
    ax.grid(alpha=0.2)
    
    ax = axes[1]
    tail = min(80, len(epochs))
    ax.semilogy(epochs[-tail:], hist256['train_losses'][-tail:], lw=1, alpha=0.7, color='#3498db')
    ax.semilogy(epochs[-tail:], hist256['val_losses'][-tail:], lw=1.5, color='#e74c3c')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss (log)')
    ax.set_title(f'Last {tail} Epochs (log scale)')
    ax.grid(alpha=0.2)
    
    fig.suptitle('AE-256d Training Curves', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Training history not found. Run train_ae256.py first.')

### 3.3 Reconstruction Quality: Side-by-Side Comparison

In [ ]:
def prepare_for_ae(X, scaler_mean, scaler_std):
    lo = float(np.percentile(X, 1))
    hi = float(np.percentile(X, 99))
    X_c = np.clip(X, lo, hi)
    return (X_c - scaler_mean) / scaler_std

def reconstruct_batch(model, X, sm, ss):
    X_norm = prepare_for_ae(X, sm, ss)
    X_t = torch.from_numpy(X_norm).unsqueeze(1).to(DEVICE)
    with torch.no_grad():
        recon_t, _ = model(X_t)
    return recon_t.cpu().numpy()[:, 0, :], X_norm

# Select diverse samples
rng = np.random.RandomState(42)
n_show = 8
sample_idx = rng.choice(len(X_clean), n_show, replace=False)
X_sample = X_clean[sample_idx]

# Reconstruct with both models
recon64, X_norm64 = reconstruct_batch(model64, X_sample, info64['scaler_mean'], info64['scaler_std'])
if has_256:
    recon256, X_norm256 = reconstruct_batch(model256, X_sample, info256['scaler_mean'], info256['scaler_std'])
    
    n_cols = 3
    fig, axes = plt.subplots(n_show, n_cols, figsize=(14, 2.5 * n_show))
    
    for i in range(n_show):
        # Original
        ax = axes[i, 0]
        ax.plot(wave, X_norm64[i], lw=1, color='#2c3e50')
        if i == 0: ax.set_title('Original (normalized)')
        ax.set_ylabel(f'#{sample_idx[i]}', fontsize=8)
        ax.grid(alpha=0.15)
        
        # AE-64d
        ax = axes[i, 1]
        ax.plot(wave, X_norm64[i], lw=0.4, alpha=0.4, color='gray')
        ax.plot(wave, recon64[i], lw=1, color='#e74c3c')
        mse = np.mean((X_norm64[i] - recon64[i])**2)
        if i == 0: ax.set_title(f'AE-64d')
        ax.set_title(f'MSE={mse:.4f}', fontsize=8, color='#e74c3c')
        ax.grid(alpha=0.15)
        
        # AE-256d
        ax = axes[i, 2]
        ax.plot(wave, X_norm256[i], lw=0.4, alpha=0.4, color='gray')
        ax.plot(wave, recon256[i], lw=1, color='#2ecc71')
        mse = np.mean((X_norm256[i] - recon256[i])**2)
        if i == 0: ax.set_title(f'AE-256d')
        ax.set_title(f'MSE={mse:.4f}', fontsize=8, color='#2ecc71')
        ax.grid(alpha=0.15)
    
    fig.suptitle('AE-64d vs AE-256d: Reconstruction Quality', fontsize=13, y=1.005)
    plt.tight_layout()
    plt.show()

### 3.4 Per-Pixel Reconstruction Error: CN Band Focus

MSE per wavelength pixel. The CN band regions (3830-3883, 4120-4216, 4285-4315) are critical.
A model that preserves CN features should have lower error in these regions relative to its overall error.

In [ ]:
def compute_per_pixel_mse(model, X, sm, ss, n_sample=5000):
    idx = np.random.RandomState(42).choice(len(X), min(n_sample, len(X)), replace=False)
    recon, X_norm = reconstruct_batch(model, X[idx], sm, ss)
    return np.mean((X_norm - recon)**2, axis=0)

pixel_mse64 = compute_per_pixel_mse(model64, X_clean, info64['scaler_mean'], info64['scaler_std'])
print(f'AE-64d  Global MSE={np.mean(pixel_mse64):.6f}')
print(f'  CN3839 (3830-3883A): {np.mean(pixel_mse64[30:83]):.6f}')
print(f'  CN4142 (4120-4216A): {np.mean(pixel_mse64[320:416]):.6f}')
print(f'  CH4300 (4285-4315A): {np.mean(pixel_mse64[485:515]):.6f}')
print(f'  Continuum (4200-4280A): {np.mean(pixel_mse64[400:480]):.6f}')

if has_256:
    pixel_mse256 = compute_per_pixel_mse(model256, X_clean, info256['scaler_mean'], info256['scaler_std'])
    print(f'\nAE-256d Global MSE={np.mean(pixel_mse256):.6f}')
    print(f'  CN3839: {np.mean(pixel_mse256[30:83]):.6f}')
    print(f'  CN4142: {np.mean(pixel_mse256[320:416]):.6f}')
    print(f'  CH4300: {np.mean(pixel_mse256[485:515]):.6f}')
    print(f'  Continuum: {np.mean(pixel_mse256[400:480]):.6f}')

# Plot comparison
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(wave, pixel_mse64, lw=1.2, color='#e74c3c', label='AE-64d (64-dim)')
if has_256:
    ax.plot(wave, pixel_mse256, lw=1.2, color='#2ecc71', label='AE-256d (256-dim)')

for band_name, (x1, x2), color in [
    ('CN3839', (3830, 3883), '#3498db'),
    ('CN4142', (4120, 4216), '#2ecc71'),
    ('CH4300', (4285, 4315), '#e67e22'),
]:
    ax.axvspan(x1, x2, alpha=0.08, color=color)
    y_pos = ax.get_ylim()[1] * 0.92 if has_256 else 0.45
    ax.text((x1+x2)/2, y_pos, band_name, ha='center', fontsize=7, color=color)

ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('MSE per pixel')
ax.set_title('Per-Pixel Reconstruction Error: AE-64d vs AE-256d')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Bottleneck Feature Extraction

Use the trained encoder to compress all 33,589 spectra into bottleneck vectors,
which serve as input features for downstream PU-Bagging classification.

In [ ]:
feats64_path = Path('SpectraAE/_cache/ae_features_64d.npy')
feats256_path = Path('SpectraAE/_cache/ae_features_256d.npy')

if feats64_path.exists():
    ae64_feats = np.load(feats64_path).astype(np.float32)
    print(f'AE-64d features: {ae64_feats.shape}')
else:
    ae64_feats = extract_features(model64, X_clean,
        scaler_mean=info64['scaler_mean'], scaler_std=info64['scaler_std'], device=DEVICE)
    np.save(feats64_path, ae64_feats)

if has_256 and feats256_path.exists():
    ae256_feats = np.load(feats256_path).astype(np.float32)
    print(f'AE-256d features: {ae256_feats.shape}')
elif has_256:
    ae256_feats = extract_features(model256, X_clean,
        scaler_mean=info256['scaler_mean'], scaler_std=info256['scaler_std'], device=DEVICE)
    np.save(feats256_path, ae256_feats)

print(f'\nAE-64d  stats: mean={ae64_feats.mean():.4f}  std={ae64_feats.std():.4f}')
if has_256:
    print(f'AE-256d stats: mean={ae256_feats.mean():.4f}  std={ae256_feats.std():.4f}')

### 4.1 Latent Space PCA Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, feats, title in [
    (axes[0], ae64_feats, 'AE-64d Latent Space (PCA)'),
    (axes[1], ae256_feats, 'AE-256d Latent Space (PCA)') if has_256 else (None, None, None),
]:
    if ax is None: continue
    pca = PCA(n_components=2).fit(feats)
    z_pca = pca.transform(feats)
    
    bg = np.random.RandomState(42).choice(len(z_pca), min(5000, len(z_pca)), replace=False)
    teff_bg = df_model.iloc[bg]['teff'].values
    sc = ax.scatter(z_pca[bg, 0], z_pca[bg, 1], s=1, alpha=0.4, c=teff_bg,
                   cmap='RdYlBu_r', edgecolors='none')
    plt.colorbar(sc, ax=ax, label='Teff (K)', shrink=0.85)
    
    pos_bg = bg[np.isin(bg, np.where(pos_mask)[0])]
    ax.scatter(z_pca[pos_bg, 0], z_pca[pos_bg, 1], s=40, alpha=0.9,
              c='black', edgecolors='white', linewidth=0.5, marker='*', label='Known CN')
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(alpha=0.15)

fig.suptitle('Bottleneck Feature Space - PCA Projection', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. PU-Bagging Classification Comparison

Run PU-Bagging (T=500) on three feature sets: AE-256d, AE-64d, Raw Spectra 700-D.

**Evaluation Metrics**:
- **PR-AUC**: Primary ranking metric (extreme class imbalance)
- **ROC-AUC**: Classification discriminability
- **P@100**: Precision in top-100 candidates
- **|r_teff| z**: Teff parameter bias after z-score debiasing (lower is better)
- **Within-r**: Within-cluster Spearman correlation with delta_CN3839 (higher = more physically consistent)

In [ ]:
T = 500
comp_csv = Path('SpectraAE/results/pu_bagging_ae256_comparison.csv')

if comp_csv.exists():
    comp_all = pd.read_csv(comp_csv)
    print('Loaded cached PU-Bagging results')
else:
    results = {}
    
    # AE-256d
    print(f'--- PU-Bagging: AE-256d (T={T}) ---')
    X256 = StandardScaler().fit_transform(ae256_feats.astype(np.float32)).astype(np.float32)
    results['AE-256d'] = run_pu_bagging(
        X256, data_pu['y_all'], data_pu['tr_idx'], data_pu['test_idx'],
        data_pu['cluster_ids'], data_pu['df_model'], T=T, name='AE-256d')
    
    # AE-64d
    print(f'--- PU-Bagging: AE-64d (T={T}) ---')
    X64 = StandardScaler().fit_transform(ae64_feats.astype(np.float32)).astype(np.float32)
    results['AE-64d'] = run_pu_bagging(
        X64, data_pu['y_all'], data_pu['tr_idx'], data_pu['test_idx'],
        data_pu['cluster_ids'], data_pu['df_model'], T=T, name='AE-64d')
    
    # Raw Spectra
    print(f'--- PU-Bagging: Raw Spectra 700-D (T={T}) ---')
    results['Raw Spectra'] = run_pu_bagging(
        data_pu['X_spec'], data_pu['y_all'], data_pu['tr_idx'], data_pu['test_idx'],
        data_pu['cluster_ids'], data_pu['df_model'], T=T, name='Raw Spectra 700-D')
    
    # Build comparison table
    def result_row(r):
        return {
            'Feature Set': r['name'], 'Dim': r['dim'], 'T': r['T'],
            'ROC': r['ROC'], 'PR': r['PR'],
            'P@50': r['P@50'], 'P@100': r['P@100'],
            '|r_teff| raw': r['|r_teff| raw'], '|r_teff| z': r['|r_teff| z'],
            'Mean bias raw': r['Mean bias raw'], 'Mean bias z': r['Mean bias z'],
            'Within-r': r['Within-r'], 'Stability': r['Stability'], 'Time': r['Time'],
        }
    comp_all = pd.DataFrame([result_row(results[k]) for k in ['AE-256d', 'AE-64d', 'Raw Spectra']])
    comp_all.to_csv(comp_csv, index=False)
    print('Saved comparison to', str(comp_csv))

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 280)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(f'\n{"="*75}')
print('FULL COMPARISON')
print(f'{"="*75}')
print(comp_all.to_string(index=False))

### 5.1 Key Metrics Visualization

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

methods = comp_all['Feature Set'].values
color_map = {'AE-64d': '#e74c3c', 'AE-256d': '#2ecc71', 'Raw Spectra 700-D': '#3498db'}
bar_colors = [color_map.get(m, '#95a5a6') for m in methods]

for ax, metric, title in zip(axes,
    ['PR', 'Within-r', '|r_teff| z', 'P@100'],
    ['PR-AUC (higher=better)', 'Within-Cluster r (higher=better)',
     '|r_teff| z-score (lower=better)', 'P@100 (higher=better)']):
    vals = comp_all[metric].values
    bars = ax.barh(range(len(methods)), vals, color=bar_colors, edgecolor='white', height=0.55)
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels([m[:22] for m in methods], fontsize=9)
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.2)
    for bar, val in zip(bars, vals):
        offset = 0.005 if val > 0.01 else 0.002
        ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

fig.suptitle('PU-Bagging: 3-Way Feature Comparison', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Probability Distribution Comparison

Compare how each feature set separates known CN stars from unlabeled stars.

In [ ]:
probs_path = Path('SpectraAE/results/ae256_pu_probs.csv')
if not probs_path.exists():
    probs_path = Path('SpectraAE/results/ae_pu_probs.csv')

if probs_path.exists():
    probs_df = pd.read_csv(probs_path)
    
    prob_sets = []
    for col, name, color in [
        ('ae256_prob', 'AE-256d', '#2ecc71'),
        ('ae64_prob', 'AE-64d', '#e74c3c'),
        ('spec_prob', 'Raw Spectra', '#3498db'),
    ]:
        if col in probs_df.columns:
            prob_sets.append((name, probs_df[col].values, color))
    
    n_sets = len(prob_sets)
    if n_sets > 0:
        fig, axes = plt.subplots(1, n_sets, figsize=(5.5 * n_sets, 4.5))
        if n_sets == 1: axes = [axes]
        
        for ax, (name, probs, color) in zip(axes, prob_sets):
            ax.hist(probs[~pos_mask], bins=80, alpha=0.7, density=True, color='#95a5a6',
                    label=f'Unlabeled')
            ax.hist(probs[pos_mask], bins=25, alpha=0.9, density=True, color=color,
                    label=f'Known CN')
            known_mean = probs[pos_mask].mean()
            ax.axvline(x=known_mean, color=color, linestyle='--', alpha=0.7,
                      label=f'CN mean={known_mean:.3f}')
            ax.set_xlabel('PU-Bagging Probability')
            ax.set_ylabel('Density')
            ax.set_title(f'{name}')
            ax.legend(fontsize=7)
            ax.grid(alpha=0.2)
        
        fig.suptitle('PU-Bagging Probability Distributions', fontsize=13, y=1.01)
        plt.tight_layout()
        plt.show()
else:
    print('No cached probability file found. Run PU-Bagging first.')

## 6. Candidate Star Analysis

Examine top candidates from AE feature sets: spectra, parameter space, and CN band indices.

In [ ]:
from ML.utils import plot_candidate_spectra, plot_teff_logg_distribution

unl_mask = ~pos_mask
top_n = 30

# Use AE-256d probs if available, else AE-64d
cand_col = 'ae256_prob' if 'ae256_prob' in (probs_df.columns if probs_path.exists() else []) else 'ae64_prob'
cand_probs = probs_df[cand_col].values if probs_path.exists() else ae64_feats.mean(axis=1)

top_local = np.argsort(cand_probs[unl_mask])[::-1][:top_n]
top_global = np.where(unl_mask)[0][top_local]

candidates = pd.DataFrame({
    'rank': range(1, top_n + 1),
    'prob': cand_probs[top_global],
    'teff': df_model.iloc[top_global]['teff'].values,
    'logg': df_model.iloc[top_global]['logg'].values,
    'feh': df_model.iloc[top_global]['feh'].values,
}).set_index('rank')

for col in ['CN3839', 'CN4142', 'CH4300', 'delta_CN3839', 'delta_CN4142', 'delta_CH4300']:
    if col in df_model.columns:
        candidates[col] = df_model.iloc[top_global][col].values

print(f'Top {top_n} Candidates:')
display(candidates.head(15).style
    .background_gradient(subset=['prob'], cmap='Reds')
    .format('{:.4f}'))

### 6.1 Top 12 Candidate Spectra vs Cluster Median

In [ ]:
cand_for_plot = candidates.copy()
cand_for_plot.index = top_global

fig = plot_candidate_spectra(
    cand_for_plot, X_clean, stars_clean, wave,
    top_n=12, prob_col='prob',
    title='Top 12 Candidates: AE Bottleneck Features + PU-Bagging'
)
plt.show()

### 6.2 Teff-Logg Parameter Space

In [ ]:
fig = plot_teff_logg_distribution(
    cand_for_plot, stars_clean, prob_col='prob',
    title='Top 30 Candidates in Teff-Logg Space'
)
plt.show()

## 7. Summary and Conclusions

### Performance Ranking

| Method | Features | PR-AUC | Within-r | |r_teff| z | Time |
|--------|----------|--------|----------|-----------|------|
| **Raw Spectra + PU-Bagging** | 700 | **0.848** | 0.244 | 0.043 | ~270s |
| CN 9-D + Focal Loss | 9 | 0.259 | 0.456 | 0.024 | <1s |
| CN 9-D + PU-Bagging | 9 | 0.180 | 0.450 | 0.040 | ~26s |
| AE-256d + PU-Bagging | 256 | (see table above) | | | |
| AE-64d + PU-Bagging | 64 | 0.167 | 0.236 | 0.007 | ~38s |

### Key Findings

1. **MSE reconstruction != CN detection**: The AE optimizes global pixel MSE,
   dominated by continuum shape (99% variance). CN band variations (~0.5% variance)
   contribute negligibly to the loss. Good reconstruction does not guarantee good downstream classification.

2. **256d vs 64d**: The 256-dim bottleneck preserves more capacity (2.7x compression vs 10.9x).
   The key question is whether the extra dimensions capture CN-band-specific information
   or just more continuum detail.

3. **Information bottleneck**: Raw spectra retain ALL information. Any compression
   necessarily loses some discriminative features. The tradeoff is speed vs accuracy.

### Improvement Directions
- **CN-band-aware loss**: Weight CN band regions (3830-3883, 4120-4216, 4285-4315) higher in MSE loss
- **Variational AE (VAE)**: Probabilistic latent space with KL regularization
- **Masked Spectrum Modeling**: MAE-style self-supervised pretraining on full dataset
- **Contrastive learning**: Learn augmentation-invariant representations

## 8. Deep Learning Training Plan Reference

The core challenge: only ~73 known positive samples (58 in training split).

---

### Strategy A: Masked Spectrum Modeling (MSM) [Recommended First]

MAE/BERT paradigm applied to 1D spectra: randomly mask 30-50% pixels,
train encoder to reconstruct masked pixels from visible ones.
Pure unsupervised, utilizes all 33,589 spectra.

**Steps**:
1. Mask strategy: 30% random mask (CN band regions masked less: 10%)
2. Encoder: 1D Swin Transformer or ConvNeXt-1D, output [CLS] or global pooling
3. Pretrain: all 33,589 spectra, MSE reconstruction, epochs=500+
4. Fine-tune: add MLP head on frozen encoder, Focal Loss or PU-Learning

**Pros**: Unsupervised, full data utilization. Learned features more semantic than AE.
**Risk**: Compute-heavy (Transformer needs GPU), mask strategy needs tuning.

---

### Strategy B: PU-Learning + Deep Neural Network [Quick Test]

Directly replace XGBoost with NN in PU-Bagging framework.

**Steps**:
1. Augment 73 positives: noise (sigma=0.001-0.01), RV jitter (+/-10 km/s),
   continuum tilt (+/-2%). Augmentation factor ~20x -> ~1,400 augmented positives.
2. Negative sampling: stratify by Teff-logg, sample 3-5x positives per epoch
3. Model: SpectraResNet + SE attention (existing code in Deep/models/resnet1d.py)
4. Loss: Focal Loss (gamma=1.5, alpha=0.95) + consistency regularization
5. Training: CosineAnnealing + SWA, epochs=200+, ensemble of 5 seeds

**Pros**: Existing infrastructure (Deep/pu_improved.py), quick iteration.
**Risk**: Augmentation may introduce unrealistic features, overfitting remains high.

---

### Strategy C: Contrastive Pretraining

SimCLR-style: two augmented versions of the same spectrum = positive pair.

**Steps**:
1. Augmentations: (a) Gaussian noise (b) wavelength shift +/-2A
   (c) local continuum scaling (d) random 5% pixel masking
2. Model: same encoder + projection head (256->128->64)
3. Loss: NT-Xent (InfoNCE), temperature tau=0.1, batch_size=512+
4. Downstream: freeze encoder, run XGBoost PU-Bagging on bottleneck features

**Pros**: Theoretically learns more discriminative features.
**Risk**: Negative sampling critical, needs large batch size (GPU memory).

---

### Strategy D: Knowledge Distillation + Pseudo-labeling

Use Raw Spectra PU-Bagging (PR=0.848) as teacher, distill to deep student.

**Steps**:
1. Pseudo-labels: select ~100-200 samples with PU-Bagging prob > 0.7
2. Confidence filter: keep z-score > 2.5 and stable within-cluster probabilities
3. Student training: CNN/Transformer with BCE+Focal Loss on expanded pseudo-labeled set
4. Iteration: regenerate pseudo-labels every 5 epochs (self-training)

**Pros**: Leverages best XGBoost results to guide DL model.
**Risk**: Pseudo-label bias propagates to student; needs careful thresholding.

---

### Recommended Implementation Path

| Priority | Strategy | Expected PR | Dev Time | Risk |
|----------|----------|-------------|----------|------|
| **1** | B: PU-Learning + NN | 0.3-0.5 | 1-2 days | Medium |
| **2** | A: MSM Pretraining | 0.4-0.7 | 1 week | Low |
| **3** | D: Knowledge Distillation | 0.5-0.7 | 3-5 days | Medium |
| **4** | C: Contrastive Learning | 0.3-0.6 | 1 week | High |

**Start with Strategy B** (reuse Deep/pu_improved.py + resnet1d.py)
for quick NN feasibility test.
**Simultaneously launch Strategy A** (MSM) as the long-term solution,
as unsupervised pretraining + full dataset is the most promising path to surpass XGBoost.

### Key Considerations

1. **Evaluation**: Always use the same 85/15 stratified split for comparability with XGBoost baseline
2. **Overfitting**: DL models easily overfit on 58 positives; use validation PR for early stopping
3. **Bias check**: After each training, check |r_teff| z and within-r to ensure model learns CN signal
4. **Progressive validation**: Test on CN 9-D features first (fast), then extend to full spectra

### Existing Resources

- `Deep/models/resnet1d.py` - SpectraResNet+SE architecture
- `Deep/pu_improved.py` - NN PU-Learning trainer
- `Deep/augmentation.py` - Spectrum augmentation utilities
- `BinaryClassifier/` - Ensemble training + evaluation framework
- `ML/pu_bagging.py` - XGBoost PU-Bagging baseline implementation